In [2]:
import pandas as pd
import os
import sys
sys.path.append('../utils')
import config_handling as conf
import database as db


In [4]:
config = conf.read_config('../config/automotive.conf.ini')
basedir = config['settings']['image_directory']
augment_csv_dump = os.path.join(basedir, 'CSV-data', 'brand phase')
basedir, db  = conf.applyconf('../config/automotive.conf.ini')

Connection established


In [57]:
testdata = pd.read_csv(os.path.join(augment_csv_dump, 'testdata_brandphase.csv'))

In [58]:
len(testdata)

798122

In [16]:
def shrinkdf(df, cols, minbound):
    return df.groupby(cols).head(minbound).reset_index(drop=True)


In [9]:
query = """SELECT 
           brand, model, year, shelltype, angletag_predicts.model_label, images.id
        FROM 
            listings
        JOIN images ON images.listing_id = listings.id
        JOIN angletag_predicts ON angletag_predicts.image_id = images.id"""

sqldata = pd.DataFrame(db.execute_query(query))

In [10]:
sqldata.sample(3)

,brand,model,year,shelltype,model_label,id
15049265,citroen,C3 (alle),2020,Stadswagen,rearleft,15710478
8679094,bmw,2 Reeks (alle),2017,Coupé,rearright,9340307
11070538,porsche,Cayenne,2024,SUV/4x4/Pick-up,rear,11731751


In [11]:
testdata.sample(5)

,image_id,model_label,model_score,yolobox_top_left_x,yolobox_top_left_y,yolobox_bottom_right_x,yolobox_bottom_right_y,bintag_predicts.image_id,model1_results,model2_results,model3_results,model4_results,brand,abs_path
295728,3323636,rearleft,1.000000,47,78,695,440,3323636,0.963563,0.998727,0.934059,0.982140,audi,/home/frederic/Documents/automotive_image_data...
561255,5282313,rearleft,0.999024,135,214,616,507,5282313,0.990428,0.999917,0.983653,0.972042,mercedes-benz,/home/frederic/Documents/automotive_image_data...
550686,7525105,frontright,0.999995,31,81,686,496,7525105,0.939091,0.983863,0.862472,0.833174,opel,/home/frederic/Documents/automotive_image_data...
522363,4577467,front,1.000000,-1,-1,-1,-1,4577467,0.546191,0.999711,0.996152,0.958433,volkswagen,/home/frederic/Documents/automotive_image_data...
272536,12760910,rear,0.998642,116,97,622,457,12760910,0.952290,1.000000,0.999531,0.999988,volkswagen,/home/frederic/Documents/automotive_image_data...


In [12]:
merged_testdata = sqldata.merge(testdata, left_on='id', right_on='image_id', how='right')

In [18]:
merged_testdata.sample(5)

,brand_x,model,year,shelltype,model_label_x,id,image_id,model_label_y,model_score,yolobox_top_left_x,yolobox_top_left_y,yolobox_bottom_right_x,yolobox_bottom_right_y,bintag_predicts.image_id,model1_results,model2_results,model3_results,model4_results,brand_y,abs_path
656529,audi,A1,2014,Berline,front,9646355,9646355,front,1.000000,35,65,711,532,9646355,0.901666,0.996726,0.934821,0.987508,audi,/home/frederic/Documents/automotive_image_data...
100700,mercedes-benz,C-Klasse (alle),2023,Break,rearright,13272553,13272553,rearright,0.999977,35,84,733,488,13272553,0.953585,0.997792,0.754347,0.213226,mercedes-benz,/home/frederic/Documents/automotive_image_data...
685688,honda,ZR-V,2024,SUV/4x4/Pick-up,frontleft,8674166,8674166,frontleft,1.000000,28,123,722,493,8674166,0.998940,1.000000,0.998534,0.999835,honda,/home/frederic/Documents/automotive_image_data...
689450,kia,Picanto,2019,Stadswagen,rear,15483287,15483287,rear,1.000000,99,98,588,491,15483287,0.973398,0.999327,0.848152,0.908712,kia,/home/frederic/Documents/automotive_image_data...
740968,toyota,C-HR,2018,SUV/4x4/Pick-up,front,8358490,8358490,front,1.000000,164,64,594,431,8358490,0.995121,0.999999,0.997993,0.995328,toyota,/home/frederic/Documents/automotive_image_data...


In [13]:
len(merged_testdata)

798122

In [14]:
len(testdata)

798122

In [25]:
merged_testdata.head(3)

,brand_x,model,year,shelltype,model_label_x,id,image_id,model_label_y,model_score,yolobox_top_left_x,yolobox_top_left_y,yolobox_bottom_right_x,yolobox_bottom_right_y,bintag_predicts.image_id,model1_results,model2_results,model3_results,model4_results,brand_y,abs_path
0,kia,Sportage,2020,SUV/4x4/Pick-up,front,8438302,8438302,front,1.000000,148,111,640,529,8438302,0.291355,0.993442,0.932474,0.432448,kia,/home/frederic/Documents/automotive_image_data...
1,ford,Transit (alle),2024,Bestelwagen,frontleft,12835404,12835404,frontleft,1.000000,13,85,741,463,12835404,0.737102,0.998978,0.798775,0.990299,ford,/home/frederic/Documents/automotive_image_data...
2,opel,Zafira Tourer,2016,Monovolume,rearleft,14772480,14772480,rearleft,0.757401,56,64,727,420,14772480,0.529626,0.999941,0.993007,0.994510,opel,/home/frederic/Documents/automotive_image_data...


In [46]:
reduced = shrinkdf(merged_testdata, ['brand_x', 'model_label_x', 'year', 'model', 'shelltype'],4)

In [47]:
merged_testdata.model_label_x.value_counts()

model_label_x
frontleft     164254
rearright     114181
frontright    112834
rearleft      100683
front          97459
rear           76241
left           72202
right          60268
Name: count, dtype: int64

In [48]:
reduced.model_label_x.value_counts()

model_label_x
frontleft     41816
frontright    37445
rearright     34854
front         33963
rearleft      33787
rear          28806
left          27492
right         25221
Name: count, dtype: int64

In [ ]:
def reducer(df, max_samples, by):
    """Training gets stuck when using the testdata during the traininprocess
    as the models get more complex; in stead of using the full testset, this 
    function will subsample it in smaller sections using a random selection
    It'll keep procentually more of smaller 'by' values 
    
    """
    sampled = []
    for _, group in df.groupby(by):
        if max_samples > len(group):
            samplesize = len(group)
        else:
            samplesize = max_samples
        small_df = group.sample(samplesize)
        sampled.append(small_df)
    return pd.concat(sampled).reset_index(drop=True)

reduced2 = reducer(testdata.query('model_label=="front"'), 1000, 'brand')

In [82]:
x='brand'
reduced2[x].nunique()

30